## Phase 2 :  

### Step 1: Generation ( X_fine and Y_Coarse)
---

In [3]:
mock_t_batch = [0.0 , 10.0]
mock_t_batch = reshape(mock_t_batch,1,1,2)
mock_k = 3

3

In [17]:

dt_fine=0.05f0
dt_coarse=0.5f0
steps = reshape(0:(mock_k-1),1,mock_k,1)
T_fine = mock_t_batch .+ ( steps .* dt_fine)
T_coarse = mock_t_batch .+ ( steps .* dt_coarse)
print(T_fine)
println(T_coarse)

[0.0 0.05000000074505806 0.10000000149011612;;; 10.0 10.050000000745058 10.100000001490116][0.0 0.5 1.0;;; 10.0 10.5 11.0]


In [18]:
function generate_tranjectories(mock_t_batch,mock_k; dt_fine=0.05f0, dt_coarse=0.5f0)

    setps = reshape(0:(mock_k-1),1,mock_k,1)
    T_fine = mock_t_batch .+ ( steps .* dt_fine)
    T_coarse = mock_t_batch .+ ( steps .* dt_coarse)

    return T_fine,T_coarse

    
end

generate_tranjectories (generic function with 1 method)

In [19]:
generate_tranjectories(mock_t_batch,mock_k)

([0.0 0.05000000074505806 0.10000000149011612;;; 10.0 10.050000000745058 10.100000001490116], [0.0 0.5 1.0;;; 10.0 10.5 11.0])

----

### Step : 3 Custom Learnable wavelet Activation Layer

$$f(x) = (w_1 \cdot \sin(x)) + (w_2 \cdot \cos(x))$$


In [22]:

# struct for WaveletActivation
struct WaveletActivation
    w1::AbstractVector
    w2::AbstractVector
end

# constrctor to initialize
WaveletActivation(d_model::Int) = WaveletActivation(ones(Float32,d_model),ones(Float32,d_model))
# @functor WaveletActivation
function (w::WaveletActivation)(x)
    return (w.w1 .* sin.(x)) .+ (w.w2 .* cos.(x))
    
end


In [23]:
d_model = 4
wavelet_layer = WaveletActivation(d_model)


WaveletActivation(Float32[1.0, 1.0, 1.0, 1.0], Float32[1.0, 1.0, 1.0, 1.0])

In [28]:
# # Generate mock input data with shape: [Features=4, Sequence=2, Batch=1]
mock_input = randn(Float32, d_model,2,1 )

4×2×1 Array{Float32, 3}:
[:, :, 1] =
  0.830001   -0.302401
  0.0150865  -1.13065
  0.749438    0.194521
 -0.949645   -0.193306

In [29]:
output_tensor = wavelet_layer(mock_input)

4×2×1 Array{Float32, 3}:
[:, :, 1] =
  1.41281    0.656811
  1.01497   -0.478616
  1.4133     1.17444
 -0.231237   0.78927

In [30]:
# ==========================================
println("=== WAVELET LAYER MOCK RUN ===")
println("Input Matrix Shape  : ", size(mock_input))
println("Output Matrix Shape : ", size(output_tensor))
println("\nOutput Array Values:")
display(output_tensor)

=== WAVELET LAYER MOCK RUN ===
Input Matrix Shape  : (4, 2, 1)
Output Matrix Shape : (4, 2, 1)

Output Array Values:


4×2×1 Array{Float32, 3}:
[:, :, 1] =
  1.41281    0.656811
  1.01497   -0.478616
  1.4133     1.17444
 -0.231237   0.78927

### Step 2 : Projectin and the postional encoding 